In [1]:
import pandas as pd
import glob
import os
import shutil

In [2]:
template_path = r"EDDTemplate\ESBasic_TRC Format.xlsx"
template_df = pd.read_excel(template_path, sheet_name="ESBasic_TRC")
template_df

,#sys_sample_code,sample_name,sample_type_code,sample_matrix_code,sample_date,sample_time,sys_loc_code,parent_sample_code,start_depth,end_depth,...,workflow_status,task_code_2,TRC_Reason_Codes,QCI,QCII,QCIII,Lab_Cert_ID_No,AltParameterCode,Approval_Code,equipment_code
0,#Text(40),Text(40),Text(20),Text(10),Date,Time,Text(20),Text(40),Numeric,Numeric,...,Text(4),Text(40),Text(50),Text(255),Text(255),Text(255),Text(255),Text(10),Text(10),Text(60)


In [6]:
dir = "data\\granger\\pfas"
# Get all Excel file paths in the directory
excel_paths = glob.glob(os.path.join(dir, "*.xlsx"))
excel_paths

['data\\granger\\pfas\\24A0011 FINAL EXCEL 24 Jan 24 0836.xlsx',
 'data\\granger\\pfas\\24A0015 FINAL EXCEL 25 Jan 24 0814.xlsx',
 'data\\granger\\pfas\\24G0039 FINAL EXCEL 23 Jul 24 1222.xlsx',
 'data\\granger\\pfas\\A16030_42_20230731154417_edits.xlsx',
 'data\\granger\\pfas\\A16030_42_EDD_20260520_163620.xlsx']

In [17]:
filename = 'A16030_42_20230731154417_edits.xlsx'
_A16030_42 = pd.read_excel(f'data\\granger\\pfas\\{filename}')
_A16030_42.shape

(352, 26)

In [10]:
pfas_chemical_dict = {
    "11Cl-PF3OUdS": "11-chloroeosafluoroundecane-1-sulfonic acid (11Cl-PF3OUdS)",
    "6:2FTS": "6:2 fluorotelomer sulfonic acid (6:2FTS)",
    "8:2FTS": "8:2 fluorotelomer sulfonic acid (8:2FTS)",
    "9Cl-PF3ONS": "9-chlorohexadecafluoro-3-oxanonane-1-sulfonic acid (9Cl-PF3ONS)",
    "ADONA": "4,8-dioxa-3H-perfluorononanoic acid (ADONA)",
    "HFPO-DA": "Hexafluoropropylene oxide dimer acid (HFPO-DA)",
    "PFBA": "Perfluorobutanoic acid (PFBA)",
    "PFBS": "Perfluorobutanesulfonic acid (PFBS)",
    "PFDA": "Perfluorodecanoic acid (PFDA)",
    "PFDoA": "Perfluorododecanoic acid (PFDoA)",
    "PFDS": "Perfluorodecanesulfonic acid (PFDS)",
    "PFHpA": "Perfluoroheptanoic acid (PFHpA)",
    "PFHpS": "Perfluoroheptanesulfonic acid (PFHpS)",
    "PFHxA": "Perfluorohexanoic acid (PFHxA)",
    "PFHXA": "Perfluorohexanoic acid (PFHxA)",
    "PFHxS": "Perfluorohexanesulfonic acid (PFHxS)",
    "PFNA": "Perfluorononanoic acid (PFNA)",
    "PFNS": "Perfluorononanesulfonic acid (PFNS)",
    "PFOA": "Perfluorooctanoic acid (PFOA)",
    "PFOS": "Perfluorooctanesulfonic acid (PFOS)",
    "PFOSA": "Perfluorooctanesulfonamide (PFOSA)",
    "PFPeA": "Perfluoropentanoic acid (PFPeA)",
    "PFPeS": "Perfluoropentanesulfonic acid (PFPeS)",
    "PFTeDA": "Perfluorotetradecanoic acid (PFTeDA)",
    "PFTrDA": "Perfluorotridecanoic acid (PFTrDA)",
    "PFUnA": "Perfluoroundecanoic acid (PFUnA)",
    "NMeFOSAA": "N-Methyl perfluorooctanesulfonamidoacetic acid  (NMeFOSAA)",
    "NEtFOSAA": "N-Ethyl perfluorooctanesulfonamidoacetic acid (NEtFOSAA)",
    "4:2FTS": "4:2 Fluorotelomer sulfonic acid (4:2 FTS)",
}

In [11]:
_A16030_42_col_dict = {'Lab ID':'lab_sample_id', 'Sample ID':"sys_loc_code", 'Analyte':'chemical_name', 
       'Result':'result_value', 'Comment':'lab_qualifiers', 'Units':'result_unit', 'Matrix':'lab_matrix',
       'Sample Collection Date':'sample_date', 'Date Extracted':'prep_date',
       'Date Analyzed':'analysis_date', 'Detection Limits':'method_detection_limit',
       'Reporting Limits':'reporting_detection_limit', 'Dilution Factor':'dilution_factor', 'CAS Number':'cas_rn',
       'Extraction Method':'prep_method'}

In [12]:
# Need to fix some of the chemical names and cas numbers
fix_pfas_cas = {"N-Methyl perfluorooctanesulfonamidoacetic acid  (NMeFOSAA)": "2355-31-9",
            "N-Ethyl perfluorooctanesulfonamidoacetic acid (NEtFOSAA)" : "2991-50-6",
            "4:2 Fluorotelomer sulfonic acid (4:2 FTS)": "757124-72-4"}

In [ ]:
_A16030_42.rename(columns=_A16030_42_col_dict, inplace=True)
# Reindex df_new using the exact column structure and order of df_template
_A16030_42_EDD = _A16030_42.reindex(columns=template_df.columns)
# update the chemical_name column using the pfas chemical dict
_A16030_42_EDD["chemical_name"] = _A16030_42_EDD["chemical_name"].map(pfas_chemical_dict).fillna(_A16030_42_EDD["chemical_name"])
# update some of the cas numbers based on fix_pfas_cas dictionary
_A16030_42_EDD["cas_rn"] = _A16030_42_EDD["chemical_name"].map(fix_pfas_cas).fillna(_A16030_42_EDD["cas_rn"])
# Convert sample_date to datetime to ensure standard formatting
sample_date_dt = pd.to_datetime(_A16030_42_EDD["sample_date"], format="%m/%d/%Y")
# copy to sample_name column to preserve original sample names before modification
_A16030_42_EDD['sample_name'] = _A16030_42_EDD['sys_loc_code']
# replace the FIELD BLANK A values in the sys_loc_code column with the corresponding sample names from the original dataframe
_A16030_42_EDD["sys_loc_code"] = _A16030_42_EDD["sys_loc_code"].replace("DUPLICATE A", "MW-6R2")
# where sample_name contains "Duplicate A", parent_sample code is "MW-6R2_20230706"
_A16030_42_EDD['parent_sample_code'] = _A16030_42_EDD['sample_name'].apply(lambda x: "MW-6R2_20230706" if "DUPLICATE A" in x else None)
# now remove the unwanted values from sys_loc_code column
# List of values you want to remove
blank_these_locs = ["EQUIPMENT BLANK", "FIELD BLANK A", "FIELD BLANK B"]
# Filter replaces the unwanted values with an empty string
_A16030_42_EDD["sys_loc_code"] = _A16030_42_EDD["sys_loc_code"].replace(blank_these_locs, "")
_A16030_42_EDD['#sys_sample_code'] = _A16030_42_EDD["sample_name"] + "_" + sample_date_dt.dt.strftime("%Y%m%d")
_A16030_42_EDD['sample_type_code'] = _A16030_42_EDD["sample_name"].apply(lambda x: "FB" if "Field Blank" in x else "N")
_A16030_42_EDD['sample_type_code'] = _A16030_42_EDD["sample_name"].apply(lambda x: "FD" if "DUPLICATE A" in x else "N")
_A16030_42_EDD["sample_matrix_code"] = _A16030_42_EDD["sample_type_code"].apply(lambda x: "WQ" if x in ("FB", "FD") else "WG")

# Addl req'd columns to be filled in:
_A16030_42_EDD['detect_flag'] = _A16030_42_EDD['result_value'].apply(lambda x: "N" if pd.isnull(x) or x == "ND" or x == pd.NaT else "Y")
_A16030_42_EDD['result_value'] = _A16030_42_EDD['result_value'].apply(lambda x: None if x in ["ND", pd.NaT] or pd.isnull(x) else x)
_A16030_42_EDD['result_unit'] = _A16030_42_EDD['result_value'].apply(lambda x: None if pd.isnull(x) else "ng/L")
_A16030_42_EDD['result_type_code'] = "TRG"
_A16030_42_EDD['reportable_result'] = "Y"
_A16030_42_EDD['test_type'] = "Initial"
_A16030_42_EDD['Lab_SDG'] = "23G0084"
_A16030_42_EDD['analytic_method'] = "E537 Mod"
_A16030_42_EDD['prep_method'] = "Method"
_A16030_42_EDD['lab_matrix'] = "WG"
_A16030_42_EDD['fraction'] = "N"
_A16030_42_EDD['detection_limit_unit'] = "ng/L"


In [15]:
_A16030_42_EDD

,#sys_sample_code,sample_name,sample_type_code,sample_matrix_code,sample_date,sample_time,sys_loc_code,parent_sample_code,start_depth,end_depth,...,workflow_status,task_code_2,TRC_Reason_Codes,QCI,QCII,QCIII,Lab_Cert_ID_No,AltParameterCode,Approval_Code,equipment_code
0,MW-6r2_20230706,MW-6r2,N,WG,07/06/2023,NaN,MW-6r2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MW-6r2_20230706,MW-6r2,N,WG,07/06/2023,NaN,MW-6r2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MW-6r2_20230706,MW-6r2,N,WG,07/06/2023,NaN,MW-6r2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MW-6r2_20230706,MW-6r2,N,WG,07/06/2023,NaN,MW-6r2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MW-6r2_20230706,MW-6r2,N,WG,07/06/2023,NaN,MW-6r2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,FIELD BLANK B_20230707,FIELD BLANK B,N,WG,07/07/2023,NaN,,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
348,FIELD BLANK B_20230707,FIELD BLANK B,N,WG,07/07/2023,NaN,,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
349,FIELD BLANK B_20230707,FIELD BLANK B,N,WG,07/07/2023,NaN,,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
350,FIELD BLANK B_20230707,FIELD BLANK B,N,WG,07/07/2023,NaN,,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# 2. Make an exact copy of the template file
# Ouptut path for the new EDD file to be created
# Make a unique name for the output file by including the original file name and a timestamp
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{dir}\\{filename.split('.')[0]}_ESBasic_{timestamp}.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    _A16030_42_EDD.to_excel(
        writer,
        sheet_name="ESBasic_TRC",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=False,  # <-- Set to False if your template already has the headers typed out
        startrow=2,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )